# Chapter 4: Deep Deterministic Policy Gradient
### **RL: The Seminal Papers** by Rahul Shirale

Welcome to the interactive companion notebook for Chapter 4. We implement **DDPG** from Lillicrap et al. (2015), "Continuous Control with Deep Reinforcement Learning." DDPG extends DQN to continuous action spaces using a deterministic actor-critic architecture with soft target updates and Ornstein-Uhlenbeck exploration noise. We verify the full pipeline on `Pendulum-v1`, which reaches a competent policy in under ten minutes on a standard CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshirale/rl-seminal-papers/blob/main/src/part_2_methods/ch04_ddpg/Chapter4_DDPG.ipynb)

## 1. Setup
The cell below installs dependencies. In Google Colab, uncomment and run it. Locally, use `make install-full` from the repo root.

In [ ]:
# Uncomment in Google Colab:
# !pip install torch>=2.0.0 gymnasium[classic-control]>=0.29.0 matplotlib numpy

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__} | Gymnasium: {gym.__version__}")

## 2. Why Continuous Actions Need a Different Approach

DQN selects actions by computing $Q(s, a)$ for every action and taking the argmax. This works when the action space is small and discrete — CartPole has two actions, Atari games have up to 18. But for a robot arm with 17 joints, each joint torque is a real number in some range $[-t_{max}, t_{max}]$. The argmax over an infinite continuous space is intractable.

DDPG's solution is a **deterministic actor** $\mu(s \mid \theta^\mu)$: a separate neural network that directly outputs the action. The critic $Q(s, a \mid \theta^Q)$ then evaluates that specific action. Training the actor reduces to maximising $Q(s, \mu(s))$ with respect to $\theta^\mu$ via the chain rule — no search over the action space required:

$$\nabla_{\theta^\mu} J \approx \mathbb{E}\left[\nabla_a Q(s, a \mid \theta^Q)\big|_{a=\mu(s)} \cdot \nabla_{\theta^\mu} \mu(s \mid \theta^\mu)\right]$$

This is the **deterministic policy gradient theorem** (Silver et al., 2014) at the heart of DDPG.

## 3. The Actor — Deterministic Policy $\mu(s \mid \theta^\mu)$

Two hidden layers of 400 and 300 units with ReLU activations, matching the paper's specification. The output passes through `tanh` and is scaled by `max_action`, so the policy always outputs actions within the environment's bounds without requiring hard clipping during training.

In [ ]:
class Actor(nn.Module):
    """
    Deterministic policy network mu(s | theta_u) from Lillicrap et al. (2015).
    Architecture: state -> FC(400) -> ReLU -> FC(300) -> ReLU -> FC(action_dim) -> tanh * max_action
    """

    def __init__(self, state_dim: int, action_dim: int, max_action: float):
        super().__init__()
        self.l1 = nn.Linear(state_dim, 400)
        self.l2 = nn.Linear(400, 300)
        self.l3 = nn.Linear(300, action_dim)
        self.max_action = max_action

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.l1(state))
        x = torch.relu(self.l2(x))
        return self.max_action * torch.tanh(self.l3(x))


# Smoke test — Pendulum-v1: state_dim=3, action_dim=1, max_action=2.0
actor_test = Actor(state_dim=3, action_dim=1, max_action=2.0)
out = actor_test(torch.zeros(1, 3))
print(f"Actor output shape: {out.shape}  (batch=1, action_dim=1)")
print(f"Output value (zero state): {out.item():.4f}  — should be in [-2.0, 2.0]")

## 4. The Critic — Action-Value Network $Q(s, a \mid \theta^Q)$

The critic estimates the expected return for taking action $a$ in state $s$. A subtle but important architectural decision from the paper: **the action is not fed in at the input layer**. Instead, the state passes through the first hidden layer alone (building a useful state representation), and the action is concatenated with that 400-unit embedding before the second hidden layer.

This gives the first layer time to extract state features before conditioning on the action, which the authors found to improve stability.

In [ ]:
class Critic(nn.Module):
    """
    Action-value network Q(s, a | theta_Q) from Lillicrap et al. (2015).
    State enters at layer 1; action enters at layer 2 after concatenation.
    Architecture: state -> FC(400) -> ReLU -> cat(action) -> FC(300) -> ReLU -> FC(1)
    """

    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.l1 = nn.Linear(state_dim, 400)
        self.l2 = nn.Linear(400 + action_dim, 300)  # action concatenated here
        self.l3 = nn.Linear(300, 1)

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.l1(state))
        x = torch.cat([x, action], dim=1)           # merge after first layer
        x = torch.relu(self.l2(x))
        return self.l3(x)


# Smoke test
critic_test = Critic(state_dim=3, action_dim=1)
q_val = critic_test(torch.zeros(1, 3), torch.zeros(1, 1))
print(f"Critic output shape: {q_val.shape}  (batch=1, scalar Q-value)")
print(f"Q-value (zero state, zero action): {q_val.item():.4f}")

## 5. Ornstein-Uhlenbeck Noise — Temporally Correlated Exploration

A deterministic policy outputs the same action for the same state, so we must add noise at evaluation time to explore. Simple Gaussian noise is independent step-to-step, which produces jittery behaviour in physical systems. The **Ornstein-Uhlenbeck (OU) process** generates noise that is correlated in time, producing smoother exploratory trajectories more suitable for inertia-based control tasks.

The discrete-time update rule is:
$$X_{t+1} = X_t + \theta(\mu - X_t) + \sigma \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, 1)$$

where $\theta = 0.15$ controls mean-reversion speed (timescale $\approx 1/\theta = 7$ steps), $\sigma = 0.2$ controls noise magnitude, and $\mu = 0$ is the long-run mean. These are the exact values reported in the paper.

In [ ]:
class OUNoise:
    """
    Ornstein-Uhlenbeck noise process for temporally correlated exploration.
    Default parameters match Lillicrap et al. (2015): theta=0.15, sigma=0.2, mu=0.
    """

    def __init__(self, size: int, mu: float = 0.0,
                 theta: float = 0.15, sigma: float = 0.2):
        self.mu    = mu * np.ones(size)
        self.theta = theta
        self.sigma = sigma
        self.reset()

    def reset(self):
        """Reset to the long-run mean at the start of each episode."""
        self.state = self.mu.copy()

    def sample(self) -> np.ndarray:
        x  = self.state
        dx = self.theta * (self.mu - x) + self.sigma * np.random.randn(len(x))
        self.state = x + dx
        return self.state.copy()


# Visualise one OU trajectory vs independent Gaussian noise
np.random.seed(0)
ou     = OUNoise(size=1)
steps  = 300
ou_traj    = [ou.sample()[0] for _ in range(steps)]
gauss_traj = list(np.random.randn(steps) * 0.2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3))

ax1.plot(ou_traj, color="#0077BB", linewidth=1.5)
ax1.axhline(0, linestyle="--", color="#999999", linewidth=0.8)
ax1.set_title("Ornstein-Uhlenbeck noise (\u03b8=0.15, \u03c3=0.2)")
ax1.set_xlabel("Step")
ax1.set_ylabel("Noise value")
ax1.grid(True, alpha=0.3)

ax2.plot(gauss_traj, color="#CC3311", linewidth=1.5)
ax2.axhline(0, linestyle="--", color="#999999", linewidth=0.8)
ax2.set_title("Independent Gaussian noise (\u03c3=0.2)")
ax2.set_xlabel("Step")
ax2.set_ylabel("Noise value")
ax2.grid(True, alpha=0.3)

plt.suptitle("OU noise is temporally correlated; Gaussian noise is i.i.d.",
             y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## 6. Replay Buffer

DDPG reuses DQN's experience replay buffer unchanged. Transitions $(s, a, r, s', \text{done})$ are stored in a circular deque of capacity 1,000,000 and sampled uniformly at random to form mini-batches. Sampling breaks the temporal correlation between consecutive transitions, satisfying gradient descent's i.i.d. assumption.

In [ ]:
import random
from collections import deque


class ReplayBuffer:
    """Circular experience replay buffer storing (s, a, r, s', done) transitions."""

    def __init__(self, max_size: int = 1_000_000):
        self.buf = deque(maxlen=max_size)

    def push(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        to_t = lambda x: torch.FloatTensor(np.array(x))
        return (
            to_t(s), to_t(a),
            to_t(r).unsqueeze(1),
            to_t(ns),
            to_t(d).unsqueeze(1),
        )

    def __len__(self) -> int:
        return len(self.buf)


# Smoke test
buf = ReplayBuffer(max_size=10_000)
for _ in range(500):
    buf.push(np.zeros(3), np.zeros(1), -1.0, np.zeros(3), False)
s, a, r, ns, d = buf.sample(64)
print(f"Buffer size: {len(buf)} | Batch — states: {s.shape}, actions: {a.shape}, rewards: {r.shape}")

## 7. Soft Target Updates

DQN periodically hard-copies online weights into the target network every $C$ steps. DDPG replaces this with a **soft update** that nudges the target network a small step toward the online network at every training step:

$$\theta' \leftarrow \tau \theta + (1 - \tau) \theta'$$

With $\tau = 0.001$, the target changes by only 0.1% per step, making the learning target extremely stable. This is one of the three key contributions of the paper (alongside replay and the deterministic policy gradient). The paper's ablation study shows that removing target networks causes training to diverge on most tasks.

## 8. The DDPG Agent

The `DDPGAgent` class assembles all five components:
- **Online actor** $\mu(s \mid \theta^\mu)$ + **target actor** $\mu'$ (frozen copy, soft-updated)
- **Online critic** $Q(s, a \mid \theta^Q)$ + **target critic** $Q'$ (frozen copy, soft-updated)
- **`OUNoise`** for exploration
- **`ReplayBuffer`** for experience storage

The `train()` method runs one mini-batch update:
1. Critic update: minimise MSE of Bellman residual using target networks for stable targets
2. Actor update: maximise $Q(s, \mu(s))$ via policy gradient (gradient ascent)
3. Soft-update both target networks

In [ ]:
class DDPGAgent:
    """
    DDPG agent from Lillicrap et al. (2015), Algorithm 1.
    Deterministic actor + action-value critic with soft target updates.
    """

    def __init__(
        self,
        state_dim:  int,
        action_dim: int,
        max_action: float,
        gamma:      float = 0.99,
        tau:        float = 0.001,
        actor_lr:   float = 1e-4,
        critic_lr:  float = 1e-3,
        batch_size: int   = 64,
        buffer_size: int  = 1_000_000,
    ):
        self.gamma      = gamma
        self.tau        = tau
        self.batch_size = batch_size
        self.max_action = max_action

        # Online networks
        self.actor  = Actor(state_dim, action_dim, max_action).to(device)
        self.critic = Critic(state_dim, action_dim).to(device)

        # Target networks — frozen copies, updated only via soft update
        self.actor_target  = copy.deepcopy(self.actor)
        self.critic_target = copy.deepcopy(self.critic)
        for p in self.actor_target.parameters():
            p.requires_grad = False
        for p in self.critic_target.parameters():
            p.requires_grad = False

        self.actor_opt  = torch.optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.replay = ReplayBuffer(buffer_size)
        self.noise  = OUNoise(action_dim)

    def select_action(self, state: np.ndarray,
                      explore: bool = True) -> np.ndarray:
        """Return a clipped action, optionally with OU exploration noise."""
        s_t = torch.FloatTensor(state).to(device)
        with torch.no_grad():
            action = self.actor(s_t).cpu().numpy()
        if explore:
            action += self.noise.sample()
        return np.clip(action, -self.max_action, self.max_action)

    def store(self, s, a, r, ns, done):
        self.replay.push(s, a, r, ns, done)

    def train(self):
        """
        One mini-batch update: critic step, actor step, soft target updates.
        Returns (critic_loss, actor_loss) or (None, None) if buffer not ready.
        """
        if len(self.replay) < self.batch_size:
            return None, None

        s, a, r, ns, d = [t.to(device) for t in self.replay.sample(self.batch_size)]

        # --- Critic update ---
        with torch.no_grad():
            na  = self.actor_target(ns)
            nq  = self.critic_target(ns, na)
            y   = r + self.gamma * (1 - d) * nq

        c_loss = F.mse_loss(self.critic(s, a), y)
        self.critic_opt.zero_grad()
        c_loss.backward()
        self.critic_opt.step()

        # --- Actor update (maximise Q via gradient ascent) ---
        a_loss = -self.critic(s, self.actor(s)).mean()
        self.actor_opt.zero_grad()
        a_loss.backward()
        self.actor_opt.step()

        # --- Soft target updates ---
        self._soft_update(self.actor_target,  self.actor)
        self._soft_update(self.critic_target, self.critic)

        return c_loss.item(), a_loss.item()

    def reset_noise(self):
        self.noise.reset()

    def _soft_update(self, target: nn.Module, source: nn.Module):
        for tp, sp in zip(target.parameters(), source.parameters()):
            tp.data.copy_(self.tau * sp.data + (1 - self.tau) * tp.data)


print("DDPGAgent defined.")

## 9. Training on Pendulum-v1

We test on `Pendulum-v1`, a classic continuous-control benchmark. The state is a 3-element vector $(\cos\theta,\, \sin\theta,\, \dot{\theta})$, and the single continuous action is torque $\in [-2, 2]$. The reward is approximately $-(\theta^2 + 0.1\dot{\theta}^2 + 0.001\tau^2)$, so the goal is to hold the pendulum upright at near-zero velocity and minimal torque. A well-trained agent typically achieves episode returns above $-200$.

In [ ]:
# Environment preview and action space information
env_preview = gym.make("Pendulum-v1", render_mode="rgb_array")
env_preview.reset(seed=42)
frame = env_preview.render()
env_preview.close()

env_info = gym.make("Pendulum-v1")
print("Observation space:", env_info.observation_space)
print("Action space:     ", env_info.action_space)
print(f"Max action:        {float(env_info.action_space.high[0]):.1f}")
env_info.close()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(frame)
ax.axis("off")
ax.set_title("Pendulum-v1 — swing up and balance the pole using continuous torque")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import clear_output

EPISODES    = 300
MAX_STEPS   = 200   # Pendulum-v1 default horizon
WARMUP_EPS  = 10    # collect experience before training
PRINT_EVERY = 10

env = gym.make("Pendulum-v1")
agent = DDPGAgent(
    state_dim  = env.observation_space.shape[0],
    action_dim = env.action_space.shape[0],
    max_action = float(env.action_space.high[0]),
)

returns      = []
critic_losses = []
actor_losses  = []

for ep in range(1, EPISODES + 1):
    state, _ = env.reset()
    agent.reset_noise()
    ep_return = 0.0

    for _ in range(MAX_STEPS):
        explore = ep > WARMUP_EPS
        action  = agent.select_action(state, explore=explore)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.store(state, action, reward, next_state, float(done))

        if ep > WARMUP_EPS:
            c_loss, a_loss = agent.train()
            if c_loss is not None:
                critic_losses.append(c_loss)
                actor_losses.append(a_loss)

        state     = next_state
        ep_return += reward

        if done:
            break

    returns.append(ep_return)

    if ep % PRINT_EVERY == 0:
        avg = np.mean(returns[-PRINT_EVERY:])
        clear_output(wait=True)

        window  = 20
        rolling = [
            np.mean(returns[max(0, i - window + 1): i + 1])
            for i in range(len(returns))
        ]

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(returns, alpha=0.2, color="#0077BB", label="Episode return")
        ax.plot(rolling, linewidth=2, color="#D97706", label=f"{window}-ep moving avg")
        ax.axhline(-200, linestyle="--", color="#117733", linewidth=1,
                   label="Competent threshold (\u2212200)")
        ax.set_title(
            f"Training \u2014 Episode {ep}/{EPISODES}"
            f"  |  Last-{PRINT_EVERY} avg: {avg:.0f}"
        )
        ax.set_xlabel("Episode")
        ax.set_ylabel("Return")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

env.close()
print("Training complete.")

## 10. Learning Curve and Loss Diagnostics

The raw episode return is noisy. The 20-episode moving average reveals the underlying trend. Below we also plot the critic and actor losses, which provide a diagnostic view of what the agent is learning:
- **Critic loss** (MSE of the Bellman residual) should generally decrease and stabilise as the value estimate improves.
- **Actor loss** (negative mean Q-value) should trend downward (become more negative) as the policy finds higher-value actions.

In [ ]:
window         = 20
returns_series = np.array(returns)
rolling_avg    = [
    np.mean(returns_series[max(0, i - window + 1): i + 1])
    for i in range(len(returns_series))
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Episode returns
axes[0].plot(returns_series, alpha=0.25, color="#0077BB", label="Episode return")
axes[0].plot(rolling_avg, linewidth=2, color="#D97706",
             label=f"{window}-ep moving avg")
axes[0].axhline(-200, linestyle="--", color="#117733", linewidth=1,
                label="Competent threshold")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return")
axes[0].set_title("DDPG on Pendulum-v1")
axes[0].legend(loc="upper left", fontsize=8,
               frameon=True, facecolor="white", framealpha=1.0,
               edgecolor="#cccccc")
axes[0].grid(True, alpha=0.3)

# Critic loss
smooth_c = np.convolve(critic_losses, np.ones(200) / 200, mode="valid")
axes[1].plot(smooth_c, linewidth=1.5, color="#CC3311")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("MSE loss")
axes[1].set_title("Critic Loss (Bellman residual)")
axes[1].grid(True, alpha=0.3)

# Actor loss
smooth_a = np.convolve(actor_losses, np.ones(200) / 200, mode="valid")
axes[2].plot(smooth_a, linewidth=1.5, color="#117733")
axes[2].set_xlabel("Training step")
axes[2].set_ylabel("\u2212Q (actor loss)")
axes[2].set_title("Actor Loss (negative mean Q-value)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_avg = np.mean(returns_series[-PRINT_EVERY:])
print(f"Final {PRINT_EVERY}-episode average: {final_avg:.1f}")
print(f"Competent threshold: \u2212200  |  {'PASSED' if final_avg > -200 else 'not yet reached'}")

## 11. Using the Module Files

In a production setting, import directly from the companion scripts rather than redefining classes in the notebook:

In [ ]:
# If running locally from the ch04_ddpg directory:
# from actor import Actor
# from critic import Critic
# from ou_noise import OUNoise
# from replay_buffer import ReplayBuffer
# from ddpg_agent import DDPGAgent
#
# To run the full Pendulum-v1 training from the terminal:
#   python src/part_2_methods/ch04_ddpg/train_pendulum.py
#
# To import from the package (repo root must be on PYTHONPATH):
#   from src.part_2_methods.ch04_ddpg import DDPGAgent